First, create the model. This must match the model used in the interactive training notebook.

In [1]:
import cv2
import torch
import torchvision

CATEGORIES = ['center']

device = torch.device('cuda')
model = torchvision.models.resnet18(pretrained=False)
model.fc = torch.nn.Linear(512, 2 * len(CATEGORIES))
model = model.cuda().eval().half()

Next, load the saved model.  Enter the model path you used to save.

In [2]:
model.load_state_dict(torch.load('road_following_model_merged_0337t_tanh_norbert_laptop.pth'))

<All keys matched successfully>

Convert and optimize the model using ``torch2trt`` for faster inference with TensorRT.  Please see the [torch2trt](https://github.com/NVIDIA-AI-IOT/torch2trt) readme for more details.

> This optimization process can take a couple minutes to complete. 

In [3]:
from torch2trt import torch2trt

data = torch.zeros((1, 3, 224, 224)).cuda().half()

model_trt = torch2trt(model, [data], fp16_mode=True)

Save the optimized model using the cell below

In [4]:
torch.save(model_trt.state_dict(), 'road_following_model_merged_0337t_tanh_norbert_laptop_trt.pth')

Load the optimized model by executing the cell below

In [5]:
import torch
from torch2trt import TRTModule

model_trt = TRTModule()
model_trt.load_state_dict(torch.load('road_following_model_merged_0337t_tanh_norbert_laptop_trt.pth'))

<All keys matched successfully>

Create the racecar class

In [6]:
from jetracer.nvidia_racecar import NvidiaRacecar

car = NvidiaRacecar()

Create the camera class.

In [7]:
from jetcam.csi_camera import CSICamera

camera = CSICamera(width=224, height=224, capture_fps=30)

Finally, execute the cell below to make the racecar move forward, steering the racecar based on the x value of the apex.

Here are some tips,

* If the car wobbles left and right,  lower the steering gain
* If the car misses turns,  raise the steering gain
* If the car tends right, make the steering bias more negative (in small increments like -0.05)
* If the car tends left, make the steering bias more postive (in small increments +0.05)

In [ ]:
## wersja zapasowa
from utils import preprocess
import numpy as np

STEERING_GAIN = -1.2
STEERING_BIAS = 0.0

#car.throttle = -0.45 0
car.throttle = -0.45

switch = 1

while switch:
    image = camera.read()
    image = preprocess(image).half()
    output = model_trt(image).detach().cpu().numpy().flatten()
    x = float(output[0])
    car.steering = x * STEERING_GAIN + STEERING_BIAS

In [12]:
import ipywidgets as widgets
from IPython.display import display

controller = widgets.Controller(index=0)
display(controller)

Controller()

In [13]:
from utils import preprocess
import numpy as np
import threading
import ipywidgets as widgets
from IPython.display import display
import time

last_speed_change = 0
STEERING_GAIN = -1.2
STEERING_BIAS = 0.0

switch      = False
manual_mode = False

start_button  = widgets.Button(description='START AUTO', button_style='success')
stop_button   = widgets.Button(description='STOP',       button_style='danger')
manual_button = widgets.Button(description='MANUAL',     button_style='warning')
mode_label    = widgets.Label(value='Tryb: ZATRZYMANY')

speed_slider    = widgets.FloatSlider(min=0.0, max=1.0, step=0.05, value=0.65,
                                      description='Prędkość:', readout_format='.2f')
steering_slider = widgets.FloatSlider(min=0.5, max=3.0, step=0.05, value=1.2,
                                      description='Steering:',  readout_format='.2f')

def on_start(b):
    global switch, manual_mode
    if not switch:
        switch       = True
        manual_mode  = False
        car.throttle = -speed_slider.value
        mode_label.value = 'Tryb: AUTO'
        print("Uruchomiono AUTO!")
        thread = threading.Thread(target=drive_loop)
        thread.start()

def on_stop(b):
    global switch
    switch = False
    car.steering = 0.0
    car.throttle = 0.0
    mode_label.value = 'Tryb: ZATRZYMANY'
    print("Zatrzymano!")

def on_manual(b):
    global manual_mode
    manual_mode = True
    mode_label.value = 'Tryb: MANUAL'
    print("Tryb manualny!")

start_button.on_click(on_start)
stop_button.on_click(on_stop)
manual_button.on_click(on_manual)

display(widgets.HBox([start_button, stop_button, manual_button]))
display(mode_label)
display(speed_slider)
display(steering_slider)

last_speed_change = 0

def drive_loop():
    global switch, manual_mode, last_speed_change

    while switch:
        now = time.time()

        if controller.buttons[0].value:
            manual_mode = False
            car.throttle = -speed_slider.value
            mode_label.value = 'Tryb: AUTO'

        if controller.buttons[1].value:
            switch = False
            car.steering = 0.0
            car.throttle = 0.0
            mode_label.value = 'Tryb: ZATRZYMANY'
            break

        if controller.buttons[2].value:
            manual_mode = True
            mode_label.value = 'Tryb: MANUAL'

        if now - last_speed_change > 0.5:    # cooldown 0.5s
            if controller.buttons[4].value:
                speed_slider.value = max(0.0, round(speed_slider.value - 0.05, 2))
                last_speed_change = now
            if controller.buttons[5].value:
                speed_slider.value = min(1.0, round(speed_slider.value + 0.05, 2))
                last_speed_change = now

        if manual_mode:
            steering = controller.axes[0].value * -1.0
            throttle = -controller.buttons[7].value * speed_slider.value
            brake    = controller.buttons[6].value
            car.steering = steering
            if brake > 0:
                car.throttle = brake
            else:
                car.throttle = throttle
        else:
            image  = camera.read()
            image  = preprocess(image).half()
            output = model_trt(image).detach().cpu().numpy().flatten()
            x      = float(output[0])
            car.steering = x * -steering_slider.value + STEERING_BIAS

    car.steering = 0.0
    car.throttle = 0.0

Label(value='Tryb: ZATRZYMANY')

FloatSlider(value=0.65, description='Prędkość:', max=1.0, step=0.05)

FloatSlider(value=1.2, description='Steering:', max=3.0, min=0.5, step=0.05)

In [27]:
## wersja zapasowa
from utils import preprocess
import numpy as np

STEERING_GAIN = -1.2
STEERING_BIAS = 0.0

#car.throttle = -0.45 0
car.throttle = -0.45

switch = 1

while switch:
    image = camera.read()
    image = preprocess(image).half()
    output = model_trt(image).detach().cpu().numpy().flatten()
    x = float(output[0])
    car.steering = x * STEERING_GAIN + STEERING_BIAS

KeyboardInterrupt: 